In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/08_integration/01_copilot_orchestrator.py

In [0]:
# ================================================================
# PHASE 21 — PIPELINE MONITORING & OBSERVABILITY
# ================================================================

print("=" * 70)
print("PHASE 21 — PIPELINE MONITORING & OBSERVABILITY")
print("=" * 70)

print()
print("Monitoring the production GenAI Data Analyst Copilot pipeline.")

In [0]:
# ================================================================
# IMPORTS
# ================================================================

import json
import time
from datetime import datetime

print("Imports successful.")

In [0]:
# ================================================================
# MONITORING DEPENDENCY CHECK
# ================================================================

print("=" * 70)
print("MONITORING DEPENDENCY CHECK")
print("=" * 70)

required_functions = [
    "ask_copilot",
    "run_sql_route",
    "run_hybrid_route",
    "generate_rag_answer"
]

failed_dependencies = []

for function_name in required_functions:

    available = callable(
        globals().get(function_name)
    )

    print(
        f"{'PASS' if available else 'FAIL'} - "
        f"{function_name}"
    )

    if not available:
        failed_dependencies.append(function_name)

print()
print(
    "Total dependencies:",
    len(required_functions)
)

print(
    "Failed dependencies:",
    len(failed_dependencies)
)

if failed_dependencies:

    raise RuntimeError(
        "Monitoring cannot continue. Missing functions: "
        + ", ".join(failed_dependencies)
    )

print()
print("Monitoring dependency check: PASS")

In [0]:
# ================================================================
# MONITORING TEST CASES
# ================================================================

monitoring_tests = [
    {
        "name": "SQL",
        "question": "Which region generated the highest revenue?"
    },
    {
        "name": "RAG",
        "question": "What is the discount policy?"
    },
    {
        "name": "HYBRID",
        "question": (
            "Which region generated the highest revenue "
            "and what discount policy applies there?"
        )
    }
]

print(
    "Monitoring test cases:",
    len(monitoring_tests)
)

In [0]:
# ================================================================
# EXECUTE MONITORING TESTS
# ================================================================

monitoring_results = []

for test in monitoring_tests:

    start_time = time.time()

    try:

        result = ask_copilot(
            test["question"]
        )

        elapsed_ms = round(
            (time.time() - start_time) * 1000,
            2
        )

        monitoring_results.append(
            {
                "test": test["name"],
                "question": test["question"],
                "success": result.get(
                    "success",
                    False
                ),
                "route": result.get(
                    "route"
                ),
                "error": result.get(
                    "error"
                ),
                "execution_time_ms": elapsed_ms
            }
        )

    except Exception as e:

        elapsed_ms = round(
            (time.time() - start_time) * 1000,
            2
        )

        monitoring_results.append(
            {
                "test": test["name"],
                "question": test["question"],
                "success": False,
                "route": None,
                "error": (
                    f"{type(e).__name__}: {str(e)}"
                ),
                "execution_time_ms": elapsed_ms
            }
        )

print(
    json.dumps(
        monitoring_results,
        indent=2,
        default=str
    )
)

In [0]:
# ================================================================
# PIPELINE METRICS
# ================================================================

total_tests = len(
    monitoring_results
)

successful_tests = sum(
    1
    for result in monitoring_results
    if result["success"]
)

failed_tests = (
    total_tests
    - successful_tests
)

success_rate = (
    successful_tests / total_tests * 100
    if total_tests > 0
    else 0
)

latencies = [
    result["execution_time_ms"]
    for result in monitoring_results
]

average_latency_ms = (
    sum(latencies) / len(latencies)
    if latencies
    else 0
)

max_latency_ms = (
    max(latencies)
    if latencies
    else 0
)

min_latency_ms = (
    min(latencies)
    if latencies
    else 0
)

print("=" * 70)
print("PIPELINE METRICS")
print("=" * 70)

print(
    "Total tests:",
    total_tests
)

print(
    "Successful tests:",
    successful_tests
)

print(
    "Failed tests:",
    failed_tests
)

print(
    "Success rate:",
    round(success_rate, 2),
    "%"
)

print(
    "Average latency:",
    round(average_latency_ms, 2),
    "ms"
)

print(
    "Minimum latency:",
    round(min_latency_ms, 2),
    "ms"
)

print(
    "Maximum latency:",
    round(max_latency_ms, 2),
    "ms"
)

In [0]:
# ================================================================
# ROUTE HEALTH
# ================================================================

print("=" * 70)
print("ROUTE HEALTH")
print("=" * 70)

for result in monitoring_results:

    print(
        f"{result['test']:10} | "
        f"Route: {str(result['route']):8} | "
        f"{'PASS' if result['success'] else 'FAIL'} | "
        f"{result['execution_time_ms']} ms"
    )

In [0]:
# ================================================================
# ERROR MONITORING
# ================================================================

print("=" * 70)
print("ERROR MONITORING")
print("=" * 70)

errors = [
    result
    for result in monitoring_results
    if not result["success"]
]

if not errors:

    print("No pipeline errors detected.")

else:

    for error in errors:

        print()
        print("Test:", error["test"])
        print("Route:", error["route"])
        print("Error:", error["error"])

In [0]:
# ================================================================
# PRODUCTION HEALTH
# ================================================================

production_healthy = (
    total_tests == 3
    and successful_tests == total_tests
    and all(
        result["route"] in [
            "sql",
            "rag",
            "hybrid"
        ]
        for result in monitoring_results
    )
)

print("=" * 70)
print("PRODUCTION HEALTH")
print("=" * 70)

print(
    "SQL:",
    "PASS"
    if monitoring_results[0]["success"]
    else "FAIL"
)

print(
    "RAG:",
    "PASS"
    if monitoring_results[1]["success"]
    else "FAIL"
)

print(
    "HYBRID:",
    "PASS"
    if monitoring_results[2]["success"]
    else "FAIL"
)

print()
print(
    "Production health:",
    "HEALTHY"
    if production_healthy
    else "UNHEALTHY"
)

In [0]:
# ================================================================
# MONITORING REPORT
# ================================================================

monitoring_report = {
    "timestamp": datetime.now().isoformat(),
    "total_tests": total_tests,
    "successful_tests": successful_tests,
    "failed_tests": failed_tests,
    "success_rate_percent": round(
        success_rate,
        2
    ),
    "average_latency_ms": round(
        average_latency_ms,
        2
    ),
    "min_latency_ms": round(
        min_latency_ms,
        2
    ),
    "max_latency_ms": round(
        max_latency_ms,
        2
    ),
    "production_healthy": production_healthy,
    "routes": monitoring_results
}

print("=" * 70)
print("MONITORING REPORT")
print("=" * 70)

print(
    json.dumps(
        monitoring_report,
        indent=2,
        default=str
    )
)

In [0]:
# ================================================================
# PHASE 21 — FINAL VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 21 — MONITORING VALIDATION")
print("=" * 70)

monitoring_passed = (
    production_healthy
    and total_tests == 3
    and successful_tests == 3
    and failed_tests == 0
)

print(
    "Total monitoring tests:",
    total_tests
)

print(
    "Successful tests:",
    successful_tests
)

print(
    "Failed tests:",
    failed_tests
)

print()

if monitoring_passed:

    print(
        "PHASE 21 STATUS: PASS ✓"
    )

else:

    print(
        "PHASE 21 STATUS: FAIL ✗"
    )